# Specialiserede Modeller — 4: Clustering — mønstre UDEN labels 🛍️🔍

Alt, hvad I har trænet indtil nu, havde en **facitliste**: giftig/spiselig, syg/rask,
hvilket ciffer. Men i den virkelige verden har data tit INGEN labels — bare rå
observationer. Kan en model overhovedet lære noget så? Ja: den kan finde **grupper**.
Det hedder **unsupervised learning**, og dagens algoritme er klassikeren **k-means**.

Scenariet: I er dataanalytikere for et butikscenter. I har 200 kunder med årsindkomst
og en "spending score" (1-100: hvor meget shopper de?). Chefen spørger: **"hvilke
kundetyper har vi?"** — og der findes ingen facitliste. Værsgo. 📊

> Selvkørende notebook (kræver kun Intro-ML). ⭐ = til de hurtige, 🐛 = bevidste fejl.

## Setup

In [ ]:
!pip install -q kagglehub

In [ ]:
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/Specialiserede-Modeller/hjaelpefunktioner.py

In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from hjaelpefunktioner import plot_kmeans_skridt

np.random.seed(42)

sti = kagglehub.dataset_download("vjchoudhary7/customer-segmentation-tutorial-in-python")
df = pd.read_csv(sti + "/Mall_Customers.csv")

# 🚨 Plan B hvis Kaggle driller — fjern #'et:
# df = pd.read_csv("https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/Specialiserede-Modeller/data/Mall_Customers.csv")

df.head()

# 21: k-means — find klumperne

Lad os først SE på kunderne. Vi plotter årsindkomst mod spending score:

In [ ]:
X_raa = df[["Annual Income (k$)", "Spending Score (1-100)"]].values.astype("float32")

plt.figure(figsize=(7, 5))
plt.scatter(X_raa[:, 0], X_raa[:, 1], alpha=0.7)
plt.xlabel("årsindkomst (tusind $)")
plt.ylabel("spending score")
plt.title("200 kunder — kan I se grupperne?")
plt.show()

Kig godt — der ER tydelige klumper (de fleste ser 5). Men vi vil ikke sidde og tegne
cirkler i hånden på 200 kunder... og slet ikke på 2 millioner. Vi vil have en algoritme.

## Algoritmen: så simpel, at det næsten er snyd

**k-means** finder $k$ klynger sådan her:

1. Placér $k$ **centre** tilfældigt (fx oven på $k$ tilfældige datapunkter).
2. **Tildel**: hvert punkt hører til det nærmeste center.
3. **Flyt**: hvert center flytter til gennemsnittet (middelpunktet) af sine punkter.
4. Gentag 2-3, indtil intet flytter sig.

Ingen gradienter, ingen tabsfunktion at differentiere — bare afstande og gennemsnit.
Først standardiserer vi (afstande på tværs af enheder kræver fælles skala — Intro-ML-
lektionen igen!), og så kører vi algoritmen SYNLIGT, trin for trin:

In [ ]:
X = (X_raa - X_raa.mean(axis=0)) / X_raa.std(axis=0)     # standardisering

k = 5
rng = np.random.default_rng(42)
centre = X[rng.choice(len(X), size=k, replace=False)].copy()   # trin 1: start på k tilfældige kunder

centre_historik = []
tildelinger_historik = []

for iteration in range(6):
    # trin 2: afstand fra hvert punkt til hvert center → vælg det nærmeste
    afstande = np.linalg.norm(X[:, None, :] - centre[None, :, :], axis=2)
    tildeling = afstande.argmin(axis=1)

    centre_historik.append(centre.copy())
    tildelinger_historik.append(tildeling.copy())

    # trin 3: flyt hvert center til gennemsnittet af dets punkter
    for j in range(k):
        centre[j] = X[tildeling == j].mean(axis=0)

plot_kmeans_skridt(X, centre_historik, tildelinger_historik)

Følg de sorte kryds (centrene) fra plot til plot: de vandrer ind i hver sin klump og
falder til ro — algoritmen er typisk færdig på en håndfuld iterationer. (Linjen med
`np.linalg.norm` regner alle 200×5 afstande på én gang — det er numpy-broadcasting og
må godt føles som sort magi; pointen er trin 2's "vælg nærmeste center".)

## Det professionelle værktøj: sklearn

Som altid i sklearn: samme mønster som træerne — bare uden `y`, for der ER ingen labels:

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=5, n_init=10, random_state=42)
kmeans.fit(X)                                  # bemærk: KUN X — ingen y!

plt.figure(figsize=(7, 5))
plt.scatter(X[:, 0], X[:, 1], c=kmeans.labels_, cmap="tab10", alpha=0.8)
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
            marker="X", s=250, color="black", edgecolors="white")
plt.xlabel("indkomst (standardiseret)")
plt.ylabel("spending score (standardiseret)")
plt.title("sklearn's KMeans, k = 5")
plt.show()

## Fra klynger til KUNDETYPER

Tallene 0-4 betyder ingenting i sig selv — fortolkningen er VORES arbejde. Vi hænger
klyngenumrene på kunderne og regner gennemsnit pr. klynge:

In [ ]:
df["klynge"] = kmeans.labels_

profil = df.groupby("klynge")[["Age", "Annual Income (k$)", "Spending Score (1-100)"]].mean().round(1)
profil["antal"] = df["klynge"].value_counts().sort_index()
profil

Prøv at give hver klynge et navn ud fra tallene — fx "velhavende storshoppere"
(høj indkomst + høj score) eller "forsigtige velhavere" (høj indkomst + lav score).
Det er præcis den slags segmenter, marketingafdelinger bygger kampagner på.

## Hvordan vælger man k? Albue-metoden 💪

k-means svarer ALTID med præcis $k$ klynger — også hvis $k$ er helt skævt. Et
pejlemærke: mål den samlede afstand fra punkterne til deres centre (kaldes **inertia**
— sklearn regner den ud for os) for forskellige $k$, og se hvor kurven "knækker" som
en albue: dér holder ekstra klynger op med at hjælpe for alvor.

In [ ]:
inertias = []
k_vaerdier = range(1, 11)
for k_test in k_vaerdier:
    km = KMeans(n_clusters=k_test, n_init=10, random_state=42)
    km.fit(X)
    inertias.append(km.inertia_)

plt.plot(k_vaerdier, inertias, "o-")
plt.xlabel("k (antal klynger)")
plt.ylabel("inertia (samlet afstand til centre)")
plt.title("Albue-metoden — find knækket")
plt.show()

### Opgaver

##### Opgave 21.1
Kør trin-for-trin-algoritmen igen, men med en anden start:
`rng = np.random.default_rng(0)` (og prøv flere). Ender centrene samme sted hver gang?
Hvad minder det jer om fra gradient descent-notebooken i Intro-ML?

In [ ]:
rng = np.random.default_rng(0)      # ← prøv 0, 1, 7 ...
centre = X[rng.choice(len(X), size=5, replace=False)].copy()
centre_historik, tildelinger_historik = [], []
for iteration in range(6):
    afstande = np.linalg.norm(X[:, None, :] - centre[None, :, :], axis=2)
    tildeling = afstande.argmin(axis=1)
    centre_historik.append(centre.copy())
    tildelinger_historik.append(tildeling.copy())
    for j in range(5):
        centre[j] = X[tildeling == j].mean(axis=0)
plot_kmeans_skridt(X, centre_historik, tildelinger_historik)

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Nogle starter ender i den "rigtige" 5-klynge-løsning, andre sætter sig fast i en skæv opdeling (fx to centre der deler én klump) — k-means finder et LOKALT minimum, præcis som gradient descent i opgave 7.6! Derfor kører sklearn algoritmen 10 gange fra forskellige starter (<code>n_init=10</code>) og beholder den bedste. Godt design: kend din algoritmes svaghed, og kør den flere gange.</span>

##### Opgave 21.2
Kør sklearn-KMeans med **k = 2, 3 og 8** og plot resultaterne. Beskriv med jeres egne
ord, hvad algoritmen "gør ved" kunderne, når k er for lille — og når k er for stor.

In [ ]:
for k_test in [2, 3, 8]:
    km = KMeans(n_clusters=k_test, n_init=10, random_state=42)
    km.fit(X)
    plt.scatter(X[:, 0], X[:, 1], c=km.labels_, cmap="tab10", alpha=0.8)
    plt.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
                marker="X", s=250, color="black", edgecolors="white")
    plt.title(f"k = {k_test}")
    plt.show()

<span style="color:red"><b>LØSNINGSFORSLAG:</b> k = 2/3: naturlige grupper tvinges sammen (fx smelter "storshoppere" og "sparsommelige" sammen). k = 8: naturlige grupper flækkes i kunstige delgrupper. k-means BRÆKKER altid dataene i præcis k stykker uden at protestere — den fortæller ikke selv, om opdelingen giver mening. Det skal albue-metoden (og jeres dømmekraft) gøre.</span>

##### Opgave 21.3
Udfyld albue-loopet (fit + gem `inertia_`) og aflæs: hvor sidder albuen for
kundedataene? Passer det med det antal klumper, I selv så i det første scatter-plot?

In [ ]:
inertias = []
for k_test in range(1, 11):
    km = KMeans(n_clusters=k_test, n_init=10, random_state=42)
    km.fit(X)
    inertias.append(km.inertia_)

plt.plot(range(1, 11), inertias, "o-")
plt.xlabel("k")
plt.ylabel("inertia")
plt.show()

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Kurven falder stejlt indtil k = 5 og flader derefter ud — albuen sidder ved 5, som matcher de fem synlige klumper. Bemærk at inertia ALTID falder, når k vokser (flere centre = kortere afstande) — derfor leder man efter knækket, ikke efter minimum.</span>

##### Opgave 21.4
Hvad sker der uden standardisering? Forestil jer, at indkomsten stod i **kroner** i
stedet for tusind dollars — kør cellen og se på klyngerne. Hvorfor ignorerer k-means
pludselig spending score fuldstændigt?

In [ ]:
X_kroner = X_raa.copy()
X_kroner[:, 0] = X_kroner[:, 0] * 7000        # tusind $ → kroner (ca.)

km = KMeans(n_clusters=5, n_init=10, random_state=42)
km.fit(X_kroner)

plt.scatter(X_kroner[:, 0], X_kroner[:, 1], c=km.labels_, cmap="tab10", alpha=0.8)
plt.xlabel("årsindkomst (kroner)")
plt.ylabel("spending score")
plt.title("k-means på u-standardiserede data")
plt.show()

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Klyngerne bliver lodrette striber: ren indkomst-opdeling, spending score er ignoreret. k-means bygger på AFSTANDE, og når den ene akse måles i hundredtusinder og den anden i enheder på 1-100, drukner den lille akse totalt i afstandsberegningen. Standardisering er ikke valgfri pynt i clustering — den AFGØR resultatet. (Samme lektion som Intro-ML opgave 10.9, nu i en tredje forklædning.)</span>

##### Opgave 21.5
Udfyld `groupby`-profilen, og giv hver af de 5 klynger et navn (à la "velhavende
storshoppere"). Hvilken klynge ville I sende luksus-reklamer til — og hvilken ville I
sende rabatkuponer?

In [ ]:
df["klynge"] = kmeans.labels_
profil = df.groupby("klynge")[["Age", "Annual Income (k$)", "Spending Score (1-100)"]].mean().round(1)
profil["antal"] = df["klynge"].value_counts().sort_index()
profil

<span style="color:red"><b>LØSNINGSFORSLAG:</b> De fem klassiske segmenter (klyngenumrene kan variere): (1) midt-indkomst/midt-score — "gennemsnitskunden" (største gruppe), (2) høj indkomst + høj score — "velhavende storshoppere" (luksus-reklamer!), (3) høj indkomst + lav score — "forsigtige velhavere" (svære at flytte), (4) lav indkomst + høj score — "unge shopaholics" (rabatkuponer... eller et venligt budgetråd), (5) lav indkomst + lav score — "sparsommelige". Fortolkningen og navngivningen er MENNESKE-arbejde — algoritmen leverer kun grupperne.</span>

##### Opgave 21.6 🐛
En kammerat har taget "alle talkolonnerne" med i clusteringen — og klyngerne er blevet
mærkeligt meningsløse striber. Kig på kolonnelisten: hvilken kolonne har intet at gøre i
en afstandsberegning, og hvorfor ødelægger den alt?

In [ ]:
X_ok = df[["Annual Income (k$)", "Spending Score (1-100)"]].values.astype("float32")
X_ok = (X_ok - X_ok.mean(axis=0)) / X_ok.std(axis=0)

km = KMeans(n_clusters=5, n_init=10, random_state=42)
km.fit(X_ok)

plt.scatter(X_ok[:, 0], X_ok[:, 1], c=km.labels_, cmap="tab10", alpha=0.8)
plt.xlabel("indkomst (std)")
plt.ylabel("spending score (std)")
plt.title("uden CustomerID — nu giver klyngerne mening")
plt.show()

<span style="color:red"><b>LØSNINGSFORSLAG:</b> <code>CustomerID</code> er bare et løbenummer — kunde 17 er ikke "tættere på" kunde 18 end på kunde 92 i nogen meningsfuld forstand. Men k-means ved det ikke: den regner pligtskyldigt afstande på ID'et som på alt andet, og fordi ID'erne er jævnt fordelt, trækker de klyngerne i meningsløse striber. Reglen gælder AL ML (husk 13.5 i Intro-ML — label i pixelrækken!): en model kan ikke selv vide, hvilke kolonner der betyder noget. Det er dit job.</span>

##### Opgave 21.7
I supervised learning kunne vi måle accuracy mod en facitliste. Her er der ingen. Så:
hvordan ved vi overhovedet, om VORES klynger er "rigtige"? Kan to forskellige opdelinger
begge være "gode"? Diskutér — og find på mindst ét konkret tjek, man kunne lave.

*Skriv jeres svar her:* $\dots$

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Der findes intet objektivt facit — kun mere eller mindre NYTTIGE opdelinger. To forskellige opdelinger kan sagtens begge være gode (én efter købekraft, én efter alder — til to forskellige kampagner). Konkrete tjek: (1) albue/inertia (matematisk kompakthed), (2) fortolkbarhed — kan hver klynge beskrives med én sætning?, (3) stabilitet — får man ca. samme klynger med anden seed/andet dataudsnit?, (4) virker det i praksis — reagerer segmenterne forskelligt på kampagnerne? Unsupervised learning flytter kvalitetsansvaret fra datasættet over på DIG.</span>

##### Opgave 21.8
Lav en helt anden segmentering: clustr på **Age** og **Spending Score** i stedet
(husk standardisering!). Hvilke nye grupper dukker op — og hvilket k foreslår
albue-metoden her?

In [ ]:
X2_raa = df[["Age", "Spending Score (1-100)"]].values.astype("float32")
X2 = (X2_raa - X2_raa.mean(axis=0)) / X2_raa.std(axis=0)

inertias = []
for k_test in range(1, 11):
    km2 = KMeans(n_clusters=k_test, n_init=10, random_state=42)
    km2.fit(X2)
    inertias.append(km2.inertia_)
plt.plot(range(1, 11), inertias, "o-")
plt.title("albue for alder + spending score")
plt.show()

km2 = KMeans(n_clusters=4, n_init=10, random_state=42)
km2.fit(X2)
plt.scatter(X2[:, 0], X2[:, 1], c=km2.labels_, cmap="tab10", alpha=0.8)
plt.xlabel("alder (std)")
plt.ylabel("spending score (std)")
plt.show()

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Et tydeligt mønster dukker op: høj spending score findes næsten kun blandt de UNGE (under ~40) — de ældre kunder ligger lavt og midt. Albuen er mindre skarp her (3-4 er begge rimelige) — sådan er virkelige data tit. Samme datasæt, nye features, helt ny historie: feature-valget bestemmer, hvilke mønstre man overhovedet KAN finde.</span>

##### Opgave 21.9
k-means' akilleshæl: kør den på månedataene fra Intro-ML (`make_moons`) og plot
resultatet. Hvorfor fejler den her, når jeres neurale netværk kunne? (Tænk på, hvad
"nærmeste center" betyder geometrisk.)

In [ ]:
from sklearn.datasets import make_moons

X_m, y_rigtige = make_moons(n_samples=400, noise=0.05, random_state=42)

km_m = KMeans(n_clusters=2, n_init=10, random_state=42)
km_m.fit(X_m)

fig, akser = plt.subplots(1, 2, figsize=(11, 4))
akser[0].scatter(X_m[:, 0], X_m[:, 1], c=y_rigtige, cmap="tab10", s=15)
akser[0].set_title("de RIGTIGE grupper")
akser[1].scatter(X_m[:, 0], X_m[:, 1], c=km_m.labels_, cmap="tab10", s=15)
akser[1].set_title("k-means' bud")
plt.show()

<span style="color:red"><b>LØSNINGSFORSLAG:</b> k-means deler månerne med en (næsten) ret linje midt igennem — helt forkert. Årsagen er indbygget: "hør til nærmeste center" giver pr. definition runde/konvekse klynger, og en halvmåne er alt andet end rund. Neurale netværk kunne bøje grænsen (med labels som facit!); k-means kan ikke — og den FORTÆLLER det ikke, den leverer bare glad sit forkerte svar. Kend din models antagelser (CNN'ets naboskab, k-means' rundhed...) — det er emnets røde tråd. (Til de nysgerrige: algoritmer som DBSCAN klarer måner fint.)</span>

##### Opgave 21.10 ⭐
Fuld segmentering: clustr på alle tre features (**Age, Annual Income, Spending
Score**). Kør albue-metoden, vælg k, og lav klynge-profilen med `groupby`. Kan I stadig
give hver klynge et navn? (3D kan ikke plottes direkte — profilen ER jeres billede.)

In [ ]:
X3_raa = df[["Age", "Annual Income (k$)", "Spending Score (1-100)"]].values.astype("float32")
X3 = (X3_raa - X3_raa.mean(axis=0)) / X3_raa.std(axis=0)

inertias = []
for k_test in range(1, 11):
    km3 = KMeans(n_clusters=k_test, n_init=10, random_state=42)
    km3.fit(X3)
    inertias.append(km3.inertia_)
plt.plot(range(1, 11), inertias, "o-")
plt.show()

km3 = KMeans(n_clusters=6, n_init=10, random_state=42)
km3.fit(X3)
df["klynge3"] = km3.labels_
profil3 = df.groupby("klynge3")[["Age", "Annual Income (k$)", "Spending Score (1-100)"]].mean().round(1)
profil3["antal"] = df["klynge3"].value_counts().sort_index()
profil3

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Med tre features bliver albuen blødere (5-6 er rimelige valg). Profilerne ligner de fem gamle segmenter, men nu splittet på alder — fx skiller "unge gennemsnitskunder" sig fra "ældre gennemsnitskunder". I mere end 2-3 dimensioner er tabellen (og domæneviden!) vigtigere end noget plot — virkelige kundesegmenteringer kører ofte på 50+ features.</span>

##### Opgave 21.11 ⭐
sklearn's `inertia_` er bare summen af kvadrerede afstande fra hvert punkt til dets
center. Regn den selv for jeres k=5-model (udfyld), og tjek at I rammer sklearn's tal.

In [ ]:
km = KMeans(n_clusters=5, n_init=10, random_state=42)
km.fit(X)

total = 0.0
for j in range(5):
    punkter = X[km.labels_ == j]
    center = km.cluster_centers_[j]
    total = total + ((punkter - center) ** 2).sum()

print("vores:  ", round(float(total), 2))
print("sklearn:", round(km.inertia_, 2))

<span style="color:red"><b>LØSNINGSFORSLAG:</b> <code>((punkter - center) ** 2).sum()</code> pr. klynge, lagt sammen — tallene matcher sklearn's på decimalerne. Inertia er altså bare "MSE-tankegangen" i klynge-forklædning: et mål for hvor KOMPAKTE klyngerne er. Ingen sort magi.</span>

##### Opgave 21.12
Butikscentret vil bruge jeres segmenter til målrettede tilbud og forskellige priser til
forskellige kundetyper. Diskutér: hvad kunne det bruges FORNUFTIGT til — og hvor går
den etiske grænse? (Tænk fx på klyngen "lav indkomst + høj spending score"...)

*Skriv jeres svar her:* $\dots$

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Fornuftigt: relevante anbefalinger, bedre sortiment, at opdage undertjente kundegrupper. Gråzone → rødt: målrettet at udnytte "lav indkomst + høj score"-segmentet (mennesker med presset økonomi og høj købelyst) med fx kviklåns-reklamer eller kunstige rabatter er teknisk trivielt og etisk giftigt — og pris-diskrimination på persondata er oven i købet juridisk mudret (GDPR!). Klyngerne er bare matematik; ANSVARET for hvad de bruges til, er menneskeligt. Godt spørgsmål at stille i enhver praktik: "hvem gavner denne segmentering?".</span>

# Godt gået! 🎉

I har nu lært jeres første unsupervised algoritme: k-means trin for trin, sklearn's
udgave, albue-metoden, fortolkning af klynger — og tre klassiske fælder (skalering,
meningsløse features, ikke-runde klynger).

**Værktøjskassen:** tabeldata → træer · billeder → CNN · data UDEN labels →
clustering · sekvenser → RNN (næste notebook!).